# BSAN391 - Minimum Cover EMS - Set covering - Gurobi

The city of Austin, Texas (source: Optimization in Operations Research, by Rardin) undertook a study of the positioning of its emergency medical service (EMS) vehicles. The city was divided into service districts needing EMS services, and vehicle stations selected from a list of alternatives so that as much of the population as possible would experience a quick response to calls for help. The figure below shows the fictitious map we will assume for our numerical version. Our city is divided into 20 service districts that we wish to serve from some combination of the 10 indicated possibilities for EMS stations. Each station can provide service to all $\textbf{adjacent}$ districts.

<img src="ems.png" style="width:350px;height:310px"/>

In [1]:
# Import Gurobi
from gurobipy import *

In [2]:
# Parameters
stations = [i for i in range(1, 10+1)]
districts, coverage, importance = multidict({1:[{2}, 5.2], 2:[{1,2}, 4.4], 3:[{1,3}, 7.1], 4:[{3}, 9], 5:[{3}, 6.1],
                                             6:[{2}, 5.7], 7:[{2,4}, 10], 8:[{3,4}, 12.2], 9:[{8}, 7.6], 10:[{4,6}, 20.3],
                                             11:[{4,5}, 20.3], 12:[{4,5,6}, 30.9], 13:[{4,5,7}, 12], 14:[{8,9}, 9.3],
                                             15:[{6,9}, 15.5], 16:[{5,6}, 25.6], 17:[{5,7,10}, 11], 18:[{8,9}, 5.3],
                                             19:[{9,10}, 7.9], 20:[{10}, 9.9]})

In [3]:
# Define EMS model 1: minimize number of selected stations - gurantee coverage
ems = Model("EMS_cover")

Academic license - for non-commercial use only - expires 2022-07-16
Using license file C:\Users\luisj\gurobi.lic


In [4]:
# Decision variables (binary variables, 1 if station is selected, 0 otherwise)
x = ems.addVars(stations, vtype=GRB.BINARY, name="station")

In [5]:
# Objective function: minimize number of selected stations
selectedStations = ems.setObjective(quicksum(x[j] for j in stations))

In [6]:
# Covering constraints
covering = ems.addConstrs(quicksum(x[j] for j in coverage[i]) >= 1 for i in districts)

In [7]:
# Solving IP
ems.optimize()

Gurobi Optimizer version 9.1.2 build v9.1.2rc0 (win64)
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads
Optimize a model with 20 rows, 10 columns and 37 nonzeros
Model fingerprint: 0x0ab091b1
Variable types: 0 continuous, 10 integer (10 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]
Found heuristic solution: objective 6.0000000
Presolve removed 20 rows and 10 columns
Presolve time: 0.00s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.01 seconds
Thread count was 1 (of 8 available processors)

Solution count 1: 6 

Optimal solution found (tolerance 1.00e-04)
Best objective 6.000000000000e+00, best bound 6.000000000000e+00, gap 0.0000%


In [8]:
#Print optimal objective function value
print("Minimum number of opened stations:", str(ems.objVal))

Minimum number of opened stations: 6.0


In [9]:
#Print optimal solution - selected locations
opened = []
for station in x.keys():
    if x[station].x > 0:
        opened.append(station)
print("Open stations:", opened)

Open stations: [2, 3, 5, 6, 8, 10]


In the Austin case, as in many other real instances, the straightforward covering model above proves inadequate because it calls for too many sites. Suppose that we have funds for only 4 EMS locations. How can we find the collection of 4 that $\textbf{minimizes coverage insufficiency}$ or $\textbf{(uncovered district importance)}$?
For this version of the model we need estimates of the demand or importance of covering each service district (see importance values in the multidict defined above).
For the sake of clarity, we'll define a brand new problem. 

In [10]:
# Define EMS model 2: minimize uncovered importance - open at most four stations
emstwo = Model("EMS_importance")

In [11]:
# Decision variables (binary variables, 1 if station is selected, 0 otherwise)
xtwo = emstwo.addVars(stations, vtype=GRB.BINARY, name="station")
# Decision variables (binary variables, 1 if district is uncovered, 0 otherwise)
y = emstwo.addVars(districts, vtype=GRB.BINARY, name="uncovered")

In [12]:
# New objective function: minimize total uncovered importance
uncoveredImportance = emstwo.setObjective(quicksum(y[i]*importance[i] for i in districts))

In [13]:
# New covering constraints
coveringtwo = emstwo.addConstrs(quicksum(xtwo[j] for j in coverage[i]) + y[i] >= 1 for i in districts)

In [14]:
# At most four stations
atmostfour = emstwo.addConstr(quicksum(xtwo[j] for j in stations) <= 4)

In [15]:
# Solving IP
emstwo.optimize()

Gurobi Optimizer version 9.1.2 build v9.1.2rc0 (win64)
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads
Optimize a model with 21 rows, 30 columns and 67 nonzeros
Model fingerprint: 0x6f67903c
Variable types: 0 continuous, 30 integer (30 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [4e+00, 3e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 4e+00]
Found heuristic solution: objective 235.3000000
Presolve removed 8 rows and 9 columns
Presolve time: 0.00s
Presolved: 13 rows, 21 columns, 46 nonzeros
Variable types: 0 continuous, 21 integer (21 binary)

Root relaxation: objective 2.750000e+01, 12 iterations, 0.00 seconds

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0   27.50000    0    8  235.30000   27.50000  88.3%     -    0s
H    0     0                      42.8000000   27.50000  35.7%     

In [16]:
# Print optimal objective function value
print("Minimum uncovered importance:", str(emstwo.objVal))

Minimum uncovered importance: 32.800000000000004


In [17]:
# Print optimal solution
openedtwo = []
uncovered = []
for station in xtwo.keys():
    if xtwo[station].x > 0:
        openedtwo.append(station)

for district in y.keys():
    if y[district].x > 0:
        uncovered.append(district)

print("Open stations:", openedtwo)
print("Districts uncovered:", uncovered)

Open stations: [3, 4, 5, 9]
Districts uncovered: [1, 2, 6, 9, 20]
